# Rivalry Pulse

**Analytical purpose:** How often did the USA and USSR meet in verified binary Olympic encounters, and who won?

This notebook is the official chart-specific preprocessing pipeline. Shared Olympic/geography logic lives in `common.py`.

In [1]:
from pathlib import Path
import sys

CHARTS_DIR = Path.cwd()
if CHARTS_DIR.name != 'charts':
    candidates = [p / 'preprocessing' / 'charts' for p in [Path.cwd(), *Path.cwd().parents]]
    CHARTS_DIR = next((p for p in candidates if (p / 'common.py').exists()), None)
    if CHARTS_DIR is None:
        raise RuntimeError('Run this notebook from the repository or preprocessing/charts directory.')
sys.path.insert(0, str(CHARTS_DIR))
from common import *
ensure_output_dirs()

In [2]:
import pandas as pd

df = load_rivalry_matches().copy()
df = df[(df["counts_for_pulse"] == True) & df["winner"].isin(["USA", "URS", "DRAW"]) & df["year"].isin(RIVALRY_YEARS)].copy()

df = df.rename(columns={
    "encounter_id": "EncounterId", "year": "Year", "city": "City", "sport": "Sport",
    "event": "Event", "encounter_type": "EncounterType", "usa_participant": "USAParticipant",
    "ussr_participant": "USSRParticipant", "winner": "Winner", "stage_raw": "StageRaw",
    "stage_normalized": "StageNormalized", "score_raw": "ScoreRaw",
    "usa_score": "USAScore", "ussr_score": "USSRScore", "source_url": "SourceUrl",
    "match_source_url": "MatchSourceUrl", "olympedia_result_id": "OlympediaResultId",
    "olympedia_match_id": "OlympediaMatchId",
})
df["Winner"] = df.Winner.replace({"URS": "USSR"})
df["SourceName"] = "Olympedia"
out = df[[
    "EncounterId", "Year", "City", "Sport", "Event", "EncounterType",
    "USAParticipant", "USSRParticipant", "Winner", "StageRaw", "StageNormalized",
    "ScoreRaw", "USAScore", "USSRScore",
    "SourceName", "SourceUrl", "MatchSourceUrl", "OlympediaResultId", "OlympediaMatchId",
]].sort_values(["Year", "Sport", "Event", "EncounterId"]).reset_index(drop=True)

assert out.EncounterId.is_unique
assert set(out.Winner) <= {"USA", "USSR", "DRAW"}
assert ((out.USAScore.notna()) == (out.USSRScore.notna())).all(), "Numeric scores must be paired"
path = FINAL_DIR / "rivalry_pulse.csv"
out.to_csv(path, index=False)
print(f"Wrote {path.relative_to(REPO_ROOT)}: {len(out)} rows; {out.USAScore.notna().sum()} numeric scores")
out.head()


Wrote data/final/cold_war/rivalry_pulse.csv: 184 rows; 94 numeric scores


,EncounterId,Year,City,Sport,Event,EncounterType,USAParticipant,USSRParticipant,Winner,StageRaw,StageNormalized,ScoreRaw,USAScore,USSRScore,SourceName,SourceUrl,MatchSourceUrl,OlympediaResultId,OlympediaMatchId
0,olympedia-31971,1952,Helsinki,Basketball,"Basketball, Men",team,United States,Soviet Union,USA,Final Round,Final,36 – 25,36.0,25.0,Olympedia,https://www.olympedia.org/results/31969,https://www.olympedia.org/results/31971,31969,31971
1,olympedia-31990,1952,Helsinki,Basketball,"Basketball, Men",team,United States,Soviet Union,USA,Group B,Group B,86 – 58,86.0,58.0,Olympedia,https://www.olympedia.org/results/31969,https://www.olympedia.org/results/31990,31969,31990
2,olympedia-22598,1952,Helsinki,Boxing,"Light-Welterweight, Men",individual,Chuck Adkins,Viktor Mednov,USA,Final Round,Final,Decision,NaN,NaN,Olympedia,https://www.olympedia.org/results/22596,https://www.olympedia.org/results/22598,22596,22598
3,olympedia-123175,1952,Helsinki,Wrestling,"Bantamweight, Freestyle, Men",individual,Bill Borders,Rəşid Məmmədbəyov,USSR,Round Three,Round 3,Fall (10:40),NaN,NaN,Olympedia,https://www.olympedia.org/results/123159,https://www.olympedia.org/results/123175,123159,123175
4,olympedia-123221,1952,Helsinki,Wrestling,"Featherweight, Freestyle, Men",individual,Joe Henson,Ibrahim Dadashev,USA,Round Four,Round 4,Decision (2-1),NaN,NaN,Olympedia,https://www.olympedia.org/results/123202,https://www.olympedia.org/results/123221,123202,123221
